# 07 — Forecast evaluation, configuration comparison, and game review

Run this whole notebook after saving at least one completed forecast run in 06.
It evaluates Sales, Rental, and both Eilers indices; preserves the evaluation;
and recommends a forecast configuration separately for each task and population.

## Settings and workflow

1. Choose `EVALUATION_MONTHS`, `MIN_MONTHS`, and `BRACKET_MONTH`. Defaults retain
   a 12-month horizon and require at least two observed target months per metric.
2. Empty `REFERENCE_FORECAST_RUN_ID` uses the latest completed compatible run
   for frozen actuals. Empty `FORECAST_RUN_IDS` compares all compatible completed
   runs. Explicit IDs give you full control.
3. Empty `COHORT_ID` builds a cohort from the reference actuals. Reuse the printed
   cohort ID in later executions to hold the labels, membership, splits, and
   brackets fixed. Existing saved cohorts are immutable.
4. Run evaluation and save before the optional chart cell. No test-game choice
   is required to calculate or save metrics.

## Fair comparisons

- Cohort eligibility depends on reference actuals and valid dates, never on a
  candidate's predictions. Demand uses observed target months; filled target
  months are excluded. Performance uses raw `*_actual` values.
- Sales cumulative actuals may include earlier zero-filled internal gaps under
  06's policy. `EXCLUDE_CUMULATIVE_IMPUTED_HISTORY` can exclude those labels too.
- One canonical game has one chronological split across Sales, Rent, and Eilers:
  approximately 60% oldest train, 20% validation, and 20% newest test. Release-date
  ties stay together, so proportions can differ and small cohorts can have an
  empty split. Train metrics are descriptive; 07 does not fit a model.
- By default, curves that start before their prediction month are excluded.
  These were explicitly flagged in 06 because source release dates may differ.
- Candidate calendars and prediction dates must agree with the frozen reference.
  A missing or mismatched prediction remains an expected point with no score.
- Automatic comparison requires the same Sales/Rental/Eilers table paths and
  Delta source versions, slot-count population, and date policy. Different
  similarity weights and cutoff/fallback settings can be compared.

## Metrics and brackets

MAE, RMSE, bias (`prediction - actual`), and directional accuracy are recorded
per game, overall, and per actual-based bracket, for train/validation/test.
Overall errors pool eligible game-month points; games with more observed months
contribute more points. Directional accuracy uses consecutive scored months;
zero change is a separate direction. No directional comparison yields null.

Brackets use the actual at exactly `BRACKET_MONTH`, within each task, metric,
population, and split. They remain fixed across forecast runs. Games without
that observed horizon are `not_available`; there is no fallback to a different
month. Equal actuals receive the same average-rank bracket, so very small or
tied groups need not have exact 20/60/20 sizes. All equal actuals are middle.

Validation MAE is the selection objective, with RMSE then run ID as deterministic
tie-breakers. A candidate must cover **all** expected validation points for that
task to be recommended. Incomplete candidates retain their measured errors and
coverage but remain unranked. Test errors are not used to choose weights.
Each task/metric/population gets its own recommendation; incompatible units are
never combined into a single score. Recommendations do not change production settings.

## Preserved outputs

| Table suffix under `development_042_silver_sandbox.demand_forecast` | Content |
| --- | --- |
| `forecast_evaluation_cohorts` | Immutable observed labels, bracket flags, splits, and cohort policy |
| `forecast_train_test_split` | Canonical-game split membership per cohort |
| `forecast_error_metrics` | Per-game, overall, and bracket metrics, including expected coverage |
| `forecast_selected_configurations` | Validation recommendations and full selected 06 configuration |
| `forecast_evaluation_runs` | Completion record, input/output versions, hashes, and MLflow links |

All tables preserve history. Repeating the save cell reuses its evaluation ID
and avoids duplicate rows. A completion record is published after all outputs
are verified. MLflow uses one parent run per evaluation and a child per forecast
configuration, with comparable metrics and copied weights/settings in one experiment.

## Review charts

Run the final cell to display the review controls below it. Choose an experiment
name, then a test game with a separate Sale or Rent choice. The game list updates
automatically and contains only games from that experiment. When multiple saved
forecast runs exist for the selected game, a separate forecast-run selector appears. It shows forecasts against the frozen observed
labels and the exact retained neighbors saved by 06. It does not recalculate
cutoffs from 05. Optional neighbor plots use the selected run's pinned source
version, ownership, slot filter, and historical observation cutoff.

This notebook collects only narrow, selected forecast curves to evaluate the
small game dataset, processing candidates one at a time. It does not convert
the underlying Sales/Rental/Eilers source tables to pandas. Spark/Delta/MLflow
integration still requires your Databricks runtime.


In [0]:
# Edit settings, then run the notebook in order.
FORECAST_RUN_IDS = []                  # Empty: all completed compatible runs.
REFERENCE_FORECAST_RUN_ID = ""         # Empty: latest compatible completed run.
COHORT_ID = ""                        # Empty: build; paste a saved cohort_id to reuse.
EVALUATION_MONTHS = 12
MIN_MONTHS = 2
BRACKET_MONTH = 12
VALIDATION_FRACTION = 0.20
TEST_FRACTION = 0.20
EXCLUDE_PRE_RELEASE_CURVES = True
EXCLUDE_CUMULATIVE_IMPUTED_HISTORY = False
SHOW_NEIGHBOR_CHARTS = False

performance = ["avg_coin_in_index_vs_house", "avg_theo_net_win_index_vs_house"]
MLFLOW_EXPERIMENT_PATH = "/Shared/AI_Feature_Extraction_Evaluation"
IMPLEMENTATION_VERSION = "07_evaluation_v2_fixed_cohort_2026_11_18"
FORECAST_RUNS_TABLE = "development_042_silver_sandbox.demand_forecast.forecast_runs"
COHORTS_TABLE = "development_042_silver_sandbox.demand_forecast.forecast_evaluation_cohorts"
SPLITS_TABLE = "development_042_silver_sandbox.demand_forecast.forecast_train_test_split"
METRICS_TABLE = "development_042_silver_sandbox.demand_forecast.forecast_error_metrics"
SELECTED_TABLE = "development_042_silver_sandbox.demand_forecast.forecast_selected_configurations"
EVALUATIONS_TABLE = "development_042_silver_sandbox.demand_forecast.forecast_evaluation_runs"


In [0]:
import copy
import hashlib
import json
import math
import uuid
from datetime import date, datetime, timezone
from numbers import Integral, Real
from functools import reduce
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, BooleanType, DateType,
)


def canonical_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)


def quoted_table_name(name):
    return ".".join("`" + part.replace("`", "``") + "`" for part in name.split("."))


def normalized_name(column):
    return F.regexp_replace(F.lower(F.trim(column.cast("string"))), r"\s+", " ")


def finite_double(column):
    value = column.cast("double")
    return F.when(value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")), value)


def require_columns(frame, columns, label):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing {missing}. Run the updated 06 first.")


def cohort_policy():
    return {
        "evaluation_months": int(EVALUATION_MONTHS), "minimum_observed_months": int(MIN_MONTHS),
        "bracket_month": int(BRACKET_MONTH), "validation_fraction": float(VALIDATION_FRACTION),
        "test_fraction": float(TEST_FRACTION), "exclude_pre_release_curves": bool(EXCLUDE_PRE_RELEASE_CURVES),
        "exclude_cumulative_imputed_history": bool(EXCLUDE_CUMULATIVE_IMPUTED_HISTORY),
        "performance_metrics": list(performance),
        "split_rule": "one chronological canonical-game split; release-date ties stay together",
        "bracket_rule": "fixed-month actual; average-tie percentile within split/task/population",
        "selection_rule": "validation MAE, complete expected-point coverage; RMSE then forecast ID ties",
        "metric_aggregation": "pooled eligible game-month errors; consecutive-month directions",
    }


def public_evaluation_settings():
    return {
        "forecast_run_ids": list(FORECAST_RUN_IDS), "reference_requested": REFERENCE_FORECAST_RUN_ID.strip(),
        "cohort_requested": COHORT_ID.strip(), "policy": cohort_policy(),
        "mlflow_experiment_path": MLFLOW_EXPERIMENT_PATH, "implementation_version": IMPLEMENTATION_VERSION,
        "tables": {"forecasts": FORECAST_RUNS_TABLE, "cohorts": COHORTS_TABLE,
                   "splits": SPLITS_TABLE, "metrics": METRICS_TABLE,
                   "selected": SELECTED_TABLE, "evaluations": EVALUATIONS_TABLE},
    }


for name, value in [("EVALUATION_MONTHS", EVALUATION_MONTHS), ("MIN_MONTHS", MIN_MONTHS), ("BRACKET_MONTH", BRACKET_MONTH)]:
    if isinstance(value, bool) or not isinstance(value, Integral) or value < 1:
        raise ValueError(f"{name} must be a positive integer.")
if MIN_MONTHS > EVALUATION_MONTHS or BRACKET_MONTH > EVALUATION_MONTHS:
    raise ValueError("MIN_MONTHS and BRACKET_MONTH must be <= EVALUATION_MONTHS.")
if any(isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or value <= 0
       for value in [VALIDATION_FRACTION, TEST_FRACTION]) or VALIDATION_FRACTION + TEST_FRACTION >= 1:
    raise ValueError("Validation/test fractions must be positive and sum to less than one.")
if not isinstance(FORECAST_RUN_IDS, list) or any(not isinstance(x, str) or not x.strip() for x in FORECAST_RUN_IDS):
    raise ValueError("FORECAST_RUN_IDS must be a list of nonempty forecast IDs.")
if len(FORECAST_RUN_IDS) != len(set(FORECAST_RUN_IDS)):
    raise ValueError("FORECAST_RUN_IDS contains duplicate IDs.")
if not MLFLOW_EXPERIMENT_PATH.startswith("/"):
    raise ValueError("Use an absolute MLflow workspace experiment path.")
LOADED_EVALUATION_SETTINGS = copy.deepcopy(public_evaluation_settings())

INPUT_TABLE_VERSIONS = {}


def pin_table(path):
    if path not in INPUT_TABLE_VERSIONS:
        if not spark.catalog.tableExists(path):
            raise ValueError(f"Required table {path} does not exist. Complete notebook 06 first.")
        INPUT_TABLE_VERSIONS[path] = int(
            spark.sql(f"DESCRIBE HISTORY {quoted_table_name(path)}")
            .agg(F.max("version").alias("v")).first()["v"]
        )
    return spark.read.option("versionAsOf", INPUT_TABLE_VERSIONS[path]).table(path)


manifest_df = pin_table(FORECAST_RUNS_TABLE)
require_columns(manifest_df, {"forecast_run_id", "forecast_mlflow_run_id", "forecast_config_hash",
    "configuration_json", "experiment_id", "similarity_mlflow_run_id", "similarity_config_hash",
    "status", "started_at_utc", "output_signatures_json", "output_locations_json"}, "Forecast completion records")
manifest_records = [row.asDict() for row in manifest_df.where(F.col("status") == "complete").collect()]
if not manifest_records:
    raise ValueError("No completed forecast runs. Save a run with the updated 06 first.")
RUN_MANIFESTS = {row["forecast_run_id"]: row for row in manifest_records}
if len(RUN_MANIFESTS) != len(manifest_records):
    raise ValueError("Duplicate forecast completion records; resolve before evaluation.")
RUN_CONFIGS = {run_id: json.loads(row["configuration_json"]) for run_id, row in RUN_MANIFESTS.items()}


def compatibility_signature(configuration):
    paths = configuration["source_paths"]
    versions = configuration["source_versions_resolved"]
    return {
        "source_paths": {key: paths[key] for key in ["sales", "lease", "performance"]},
        "source_versions": {key: int(versions[key]) for key in ["sales", "lease", "performance"]},
        "slot_counts": sorted(float(x) for x in configuration["slot_counts"]),
        "prediction_date_rule": configuration["prediction_date_rule"],
        "neighbor_observation_rule": configuration["neighbor_observation_rule"],
        "calendar_rule": configuration["calendar_rule"],
        "target_actual_policy": configuration["target_actual_policy"],
    }


SAVED_COHORT_RECORDS = None
SAVED_COHORT_CONFIG = None
if COHORT_ID.strip():
    existing_cohort_df = pin_table(COHORTS_TABLE).where(F.col("cohort_id") == COHORT_ID.strip())
    require_columns(existing_cohort_df, {"cohort_id", "cohort_config_json", "cohort_content_hash"}, "Saved cohort")
    SAVED_COHORT_RECORDS = [row.asDict() for row in existing_cohort_df.toLocalIterator()]
    if not SAVED_COHORT_RECORDS:
        raise ValueError(f"Unknown COHORT_ID {COHORT_ID!r}.")
    configs = {row["cohort_config_json"] for row in SAVED_COHORT_RECORDS}
    if len(configs) != 1:
        raise ValueError("Saved cohort has inconsistent policy metadata.")
    SAVED_COHORT_CONFIG = json.loads(next(iter(configs)))
    if SAVED_COHORT_CONFIG["policy"] != cohort_policy():
        raise ValueError("Current evaluation settings differ from this saved cohort. Match its policy or leave COHORT_ID empty to create another.")
    required_compatibility = SAVED_COHORT_CONFIG["forecast_compatibility"]
else:
    required_compatibility = None

ordered_runs = sorted(RUN_MANIFESTS, key=lambda run_id: (RUN_MANIFESTS[run_id]["started_at_utc"], run_id), reverse=True)
def can_supply_horizon(run_id):
    cfg = RUN_CONFIGS[run_id]
    return int(cfg["horizon_months"]) >= EVALUATION_MONTHS and set(performance).issubset(cfg["performance_metrics"])

REFERENCE_RUN_ID = REFERENCE_FORECAST_RUN_ID.strip()
if not REFERENCE_RUN_ID:
    options = [run_id for run_id in ordered_runs if can_supply_horizon(run_id)
               and (required_compatibility is None or compatibility_signature(RUN_CONFIGS[run_id]) == required_compatibility)]
    if not options:
        raise ValueError("No completed run matches the cohort settings and evaluation horizon. Check EVALUATION_MONTHS and the source snapshots.")
    REFERENCE_RUN_ID = options[0]
if REFERENCE_RUN_ID not in RUN_MANIFESTS or not can_supply_horizon(REFERENCE_RUN_ID):
    raise ValueError("The reference run is unknown, incomplete, or shorter than the requested evaluation horizon.")
REFERENCE_COMPATIBILITY = compatibility_signature(RUN_CONFIGS[REFERENCE_RUN_ID])
if required_compatibility is not None and REFERENCE_COMPATIBILITY != required_compatibility:
    raise ValueError("The reference run uses a different source population/snapshot from the saved cohort.")

compatible_ids = [run_id for run_id in ordered_runs if can_supply_horizon(run_id)
                  and compatibility_signature(RUN_CONFIGS[run_id]) == REFERENCE_COMPATIBILITY]
SELECTED_RUN_IDS = list(FORECAST_RUN_IDS) if FORECAST_RUN_IDS else compatible_ids
invalid = [run_id for run_id in SELECTED_RUN_IDS if run_id not in compatible_ids]
if invalid:
    raise ValueError(f"These forecast IDs are unknown/incomplete or use incompatible snapshots, populations, or horizons: {invalid}")
if not SELECTED_RUN_IDS:
    raise ValueError("No compatible forecast runs selected.")
# Report all completed runs so missing forecasts and incompatible snapshots
# are distinguishable from the experiment/game review filter.
selection_diagnostics = []
for candidate_id in ordered_runs:
    cfg = RUN_CONFIGS[candidate_id]
    reasons = []
    if not can_supply_horizon(candidate_id):
        reasons.append("horizon or performance metrics do not match")
    signature = compatibility_signature(cfg)
    for field in REFERENCE_COMPATIBILITY:
        if signature[field] != REFERENCE_COMPATIBILITY[field]:
            reasons.append(f"different {field}")
    if not reasons and candidate_id not in SELECTED_RUN_IDS:
        reasons.append("excluded by FORECAST_RUN_IDS")
    selection_diagnostics.append({
        "experiment_id": RUN_MANIFESTS[candidate_id]["experiment_id"],
        "forecast_run_id": candidate_id,
        "included": candidate_id in SELECTED_RUN_IDS,
        "reason": "; ".join(reasons) or "selected",
    })
print("Completed forecast runs and evaluation selection:")
display(pd.DataFrame(selection_diagnostics))
print(f"Evaluating {len({RUN_MANIFESTS[run_id]['experiment_id'] for run_id in SELECTED_RUN_IDS})} distinct similarity experiments.")
print("Experiments without completed forecasts must be run through notebook 06 first.")

LOAD_RUN_IDS = sorted(set([*SELECTED_RUN_IDS, REFERENCE_RUN_ID]))
RUN_METADATA = {
    run_id: {key: RUN_MANIFESTS[run_id][key] for key in [
        "forecast_run_id", "forecast_mlflow_run_id", "forecast_config_hash", "experiment_id",
        "similarity_mlflow_run_id", "similarity_config_hash",
    ]} for run_id in LOAD_RUN_IDS
}


def load_forecast_output(label):
    paths = {}
    for run_id in LOAD_RUN_IDS:
        location = json.loads(RUN_MANIFESTS[run_id]["output_locations_json"])[label]
        paths.setdefault(location["table"], []).append(run_id)
    frames = []
    for path, run_ids in paths.items():
        frame = pin_table(path)
        require_columns(frame, {"forecast_run_id", "forecast_mlflow_run_id", "forecast_config_hash"}, label)
        frames.append(frame.where(F.col("forecast_run_id").isin(run_ids)))
    combined = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), frames).cache()
    counts = {row["forecast_run_id"]: row["count"] for row in combined.groupBy("forecast_run_id").count().collect()}
    for run_id in LOAD_RUN_IDS:
        expected = json.loads(RUN_MANIFESTS[run_id]["output_signatures_json"])[label]["row_count"]
        if counts.get(run_id, 0) != expected:
            raise ValueError(f"{label} rows for {run_id} differ from its completed-run record.")
    identity_rows = combined.select("forecast_run_id", "forecast_mlflow_run_id", "forecast_config_hash").distinct().collect()
    for row in identity_rows:
        meta = RUN_METADATA[row["forecast_run_id"]]
        if row["forecast_mlflow_run_id"] != meta["forecast_mlflow_run_id"] or row["forecast_config_hash"] != meta["forecast_config_hash"]:
            raise ValueError(f"{label} lineage differs from the selected completion record.")
    return combined


for previous in globals().get("CACHED_EVALUATION_INPUTS", []):
    previous.unpersist()
demand_source_all_df = load_forecast_output("demand")
performance_source_all_df = load_forecast_output("performance")
forecast_neighbors_source_df = load_forecast_output("neighbors")
CACHED_EVALUATION_INPUTS = [demand_source_all_df, performance_source_all_df, forecast_neighbors_source_df]
demand_all_df = demand_source_all_df.where(F.col("forecast_run_id").isin(SELECTED_RUN_IDS))
performance_all_df = performance_source_all_df.where(F.col("forecast_run_id").isin(SELECTED_RUN_IDS))
forecast_neighbors_df = forecast_neighbors_source_df.where(F.col("forecast_run_id").isin(SELECTED_RUN_IDS))
for row in demand_source_all_df.select("forecast_run_id", "experiment_name").distinct().collect():
    RUN_METADATA[row["forecast_run_id"]]["experiment_name"] = row["experiment_name"]
for meta in RUN_METADATA.values():
    meta.setdefault("experiment_name", meta["experiment_id"])

LONG_KEYS = ["target_game_key", "forecast_type", "metric_name", "revenue_type", "own_status", "month_index"]


def long_forecast_rows(demand, perf):
    shared = ["forecast_run_id", "target_game_key", "_target_id", "target_game", "game_release_date", "prediction_date",
              "revenue_type", "own_status", "month_index", "target_curve_starts_before_prediction"]
    require_columns(demand, {*shared, "metric_name", "forecast_month", "forecast_value", "actual_value",
        "actual_is_observed", "actual_contains_imputed_history"}, "Demand curves")
    require_columns(perf, {*shared, "forecast_yearmonth",
        *[f"{metric}_{suffix}" for metric in performance for suffix in ["mean", "actual"]]}, "Performance curves")
    demand_rows = demand.select(
        *shared, F.lit("demand").alias("forecast_type"), "metric_name", "forecast_month",
        finite_double(F.col("forecast_value")).alias("prediction"),
        finite_double(F.col("actual_value")).alias("actual"),
        F.coalesce("actual_is_observed", F.lit(False)).alias("actual_is_observed"),
        F.coalesce("actual_contains_imputed_history", F.lit(False)).alias("actual_contains_imputed_history"),
    )
    frames = [demand_rows]
    for metric in performance:
        frames.append(perf.select(
            *shared, F.lit("performance").alias("forecast_type"), F.lit(metric).alias("metric_name"),
            F.to_date(F.concat(F.col("forecast_yearmonth").cast("string"), F.lit("01")), "yyyyMMdd").alias("forecast_month"),
            finite_double(F.col(f"{metric}_mean")).alias("prediction"),
            finite_double(F.col(f"{metric}_actual")).alias("actual"),
            finite_double(F.col(f"{metric}_actual")).isNotNull().alias("actual_is_observed"),
            F.lit(False).alias("actual_contains_imputed_history"),
        ))
    return reduce(lambda a, b: a.unionByName(b), frames).where(F.col("month_index").between(1, EVALUATION_MONTHS))


long_all_df = long_forecast_rows(demand_source_all_df, performance_source_all_df)
print(f"Reference forecast run: {REFERENCE_RUN_ID}")
print(f"Selected {len(SELECTED_RUN_IDS)} compatible forecast runs.")
display(manifest_df.where(F.col("forecast_run_id").isin(SELECTED_RUN_IDS)).select(
    "forecast_run_id", "experiment_id", "started_at_utc", "forecast_config_hash"))


In [0]:
# Fixed-cohort evaluation functions. These operate only on saved forecast curves
# projected to the columns below, never on full sales or Eilers source tables.
import math
from numbers import Integral, Real
import numpy as np
import pandas as pd

TASK_KEYS = ["forecast_type", "metric_name", "revenue_type", "own_status"]
SERIES_KEYS = [*TASK_KEYS, "target_game_key"]
POINT_KEYS = [*SERIES_KEYS, "month_index"]
DATE_COLUMNS = ["game_release_date", "prediction_date", "forecast_month"]


def _required_columns(frame, columns, label):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing columns: {missing}")


def _true(value):
    if value is None or pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def _positive_int(value, name):
    if isinstance(value, bool) or not isinstance(value, Integral) or value < 1:
        raise ValueError(f"{name} must be a positive integer.")


def _dates(values, month=False):
    values = pd.to_datetime(values, errors="coerce")
    if month:
        values = values.dt.to_period("M").dt.to_timestamp()
    return values.dt.date


def _canonical_keys(frame):
    frame = frame.copy()
    frame["target_game_key"] = (
        frame["target_game_key"].astype("string").str.strip().str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    return frame


def assign_temporal_splits(records, val_fraction=0.20, test_fraction=0.20):
    """One chronological game split; equal release dates always stay together.

    Test uses the newest ceil(N*test_fraction) boundary, including date ties.
    Validation uses the newest ceil(N*val_fraction) boundary strictly before
    test. Remaining older dates are train. Small/tied populations can leave a
    split empty; no train or test observations are reused to fill validation.
    """
    for name, fraction in [("val_fraction", val_fraction), ("test_fraction", test_fraction)]:
        if isinstance(fraction, bool) or not isinstance(fraction, Real) or not math.isfinite(float(fraction)) or not 0 < fraction < 1:
            raise ValueError(f"{name} must be finite and strictly between zero and one.")
    if val_fraction + test_fraction >= 1:
        raise ValueError("Validation and test fractions must sum to less than one.")
    games = [dict(row) for row in records]
    if not games:
        return []
    keys = [row["target_game_key"] for row in games]
    if len(keys) != len(set(keys)):
        raise ValueError("Temporal splitting requires one record per canonical game.")
    for row in games:
        value = pd.to_datetime(row["game_release_date"], errors="coerce")
        if pd.isna(value):
            raise ValueError("Every split game needs a valid release date.")
        row["game_release_date"] = value.date()
    games.sort(key=lambda row: (row["game_release_date"], row["target_game_key"]))
    count = len(games)
    test_count = max(1, math.ceil(count * test_fraction))
    test_cutoff = games[-test_count]["game_release_date"]
    older = [row for row in games if row["game_release_date"] < test_cutoff]
    val_count = max(1, math.ceil(count * val_fraction))
    val_cutoff = older[-min(val_count, len(older))]["game_release_date"] if older else None
    result = []
    for row in games:
        released = row["game_release_date"]
        split = "test" if released >= test_cutoff else (
            "validation" if val_cutoff is not None and released >= val_cutoff else "train"
        )
        result.append({
            **row, "split": split,
            "split_cutoff_date": test_cutoff,
            "test_cutoff_date": test_cutoff,
            "validation_cutoff_date": val_cutoff,
            "eligible_games_in_cohort": count,
        })
    return result


def build_reference_cohort(
    reference_records, evaluation_months=12, min_months=2, bracket_month=12,
    val_fraction=0.20, test_fraction=0.20, exclude_pre_release=True,
    exclude_cumulative_imputed_history=False,
):
    """Freeze observed labels, games, splits and brackets without predictions.

    Cohort eligibility is per game/task/metric and requires min_months observed
    actuals in the chosen horizon. Brackets use only the actual at bracket_month,
    within split/task/metric. Shorter histories get not_available. Average-tie
    percent ranks preserve equal actuals: <=0.2 bottom, >=0.8 top. A population
    with equal actuals is middle. Ties/small populations change bracket sizes.
    """
    for name, value in [("evaluation_months", evaluation_months), ("min_months", min_months), ("bracket_month", bracket_month)]:
        _positive_int(value, name)
    if min_months > evaluation_months or bracket_month > evaluation_months:
        raise ValueError("MIN_MONTHS and BRACKET_MONTH cannot exceed the evaluation horizon.")
    frame = pd.DataFrame(reference_records).copy()
    _required_columns(frame, [*POINT_KEYS, "_target_id", "target_game", *DATE_COLUMNS,
                             "actual", "actual_is_observed"], "Reference actuals")
    frame = _canonical_keys(frame)
    if frame["target_game_key"].isna().any() or frame["target_game_key"].eq("").any():
        raise ValueError("Reference actuals require nonempty canonical game keys.")
    for column in DATE_COLUMNS:
        frame[column] = _dates(frame[column], month=column == "forecast_month")

    # Conflicting canonical identities are an input-data error, even if one
    # task would later be excluded for insufficient actual history.
    for key, group in frame.groupby("target_game_key", sort=True):
        for column in ["game_release_date", "_target_id"]:
            if group[column].dropna().nunique() > 1:
                raise ValueError(f"Conflicting {column} values for canonical game {key!r}.")
    if frame["_target_id"].isna().any():
        raise ValueError("Reference games require stable _target_id values.")

    frame["actual"] = pd.to_numeric(frame["actual"], errors="coerce")
    frame["month_index"] = pd.to_numeric(frame["month_index"], errors="coerce")
    observed = frame["actual_is_observed"].map(_true)
    mask = observed & np.isfinite(frame["actual"]) & frame["month_index"].between(1, evaluation_months)
    mask &= frame["month_index"].mod(1).eq(0)
    mask &= frame["game_release_date"].notna() & frame["prediction_date"].notna() & frame["forecast_month"].notna()
    if exclude_pre_release:
        prior_curve = frame.get("target_curve_starts_before_prediction", pd.Series(False, index=frame.index)).map(_true)
        prior_series = prior_curve.groupby([frame[key] for key in SERIES_KEYS], dropna=False).transform("max")
        mask &= ~prior_series
        prediction_months = _dates(frame["prediction_date"], month=True)
        mask &= frame["forecast_month"] >= prediction_months
    if exclude_cumulative_imputed_history:
        imputed_history = frame.get("actual_contains_imputed_history", pd.Series(False, index=frame.index)).map(_true)
        mask &= ~((frame["forecast_type"] == "demand") & (frame["revenue_type"] == "sales") & imputed_history)
    frame = frame.loc[mask].copy()
    frame["month_index"] = frame["month_index"].astype("int64")
    if frame.duplicated(POINT_KEYS).any():
        raise ValueError("Reference actuals contain duplicate canonical game/task/month points.")
    if frame.empty:
        raise ValueError("No observed reference actuals satisfy the evaluation policy.")
    frame["reference_observed_months"] = frame.groupby(SERIES_KEYS, dropna=False)["month_index"].transform("count")
    frame = frame.loc[frame["reference_observed_months"] >= min_months].copy()
    if frame.empty:
        raise ValueError("No reference game/task has the required observed actual months.")

    # Drop any prediction columns supplied by accident; the cohort contains
    # labels and policy metadata only, and never depends on forecast coverage.
    label_columns = [*POINT_KEYS, "_target_id", "target_game", *DATE_COLUMNS,
                     "actual", "reference_observed_months"]
    frame = frame[label_columns]
    games = (
        frame.sort_values(["target_game_key", "target_game"])
        .drop_duplicates("target_game_key")
        [["target_game_key", "_target_id", "target_game", "game_release_date"]]
    )
    games = pd.DataFrame(assign_temporal_splits(games.to_dict("records"), val_fraction, test_fraction))
    frame = frame.merge(games[["target_game_key", "split"]], on="target_game_key", how="left", validate="many_to_one")
    frame["evaluation_split"] = frame["split"]

    brackets = frame.loc[frame["month_index"] == bracket_month, [*SERIES_KEYS, "split", "actual"]].copy()
    brackets = brackets.rename(columns={"actual": "bracket_actual"})
    brackets["bracket_percent_rank"] = np.nan
    brackets["actual_bracket"] = "middle_60"
    for _, group in brackets.groupby([*TASK_KEYS, "split"], sort=True, dropna=False):
        if len(group) > 1 and group["bracket_actual"].nunique() > 1:
            percentile = (group["bracket_actual"].rank(method="average") - 1) / (len(group) - 1)
            brackets.loc[group.index, "bracket_percent_rank"] = percentile
            brackets.loc[group.index[percentile <= 0.20], "actual_bracket"] = "bottom_20"
            brackets.loc[group.index[percentile >= 0.80], "actual_bracket"] = "top_20"
        else:
            brackets.loc[group.index, "bracket_percent_rank"] = 0.50
    frame = frame.merge(brackets, on=[*SERIES_KEYS, "split"], how="left", validate="many_to_one")
    frame["actual_bracket"] = frame["actual_bracket"].fillna("not_available")
    frame["bracket_month"] = int(bracket_month)
    return (
        frame.sort_values([*SERIES_KEYS, "month_index"]).reset_index(drop=True),
        games.sort_values(["game_release_date", "target_game_key"]).reset_index(drop=True),
    )


def evaluate_forecast_run(prediction_records, points_df, min_months=2, metadata=None):
    """Left-align one candidate to frozen labels and retain missing predictions.

    MAE/RMSE/bias pool observed month pairs, not game means. Directional
    accuracy compares only consecutive scored months within a game/task;
    unchanged values are a third direction category. Validation selection
    requires complete coverage of all expected points and games.
    """
    _positive_int(min_months, "min_months")
    _required_columns(points_df, [*POINT_KEYS, "_target_id", "target_game", "actual",
                                 "forecast_month", "prediction_date", "evaluation_split", "actual_bracket"], "Frozen cohort")
    points = points_df.copy().reset_index(drop=True)
    if points.duplicated(POINT_KEYS).any():
        raise ValueError("Frozen cohort has duplicate expected points.")
    metadata = dict(metadata or {})
    overlap = set(metadata) & set(points.columns)
    if overlap:
        raise ValueError(f"Run metadata cannot replace frozen label fields: {sorted(overlap)}")
    predictions = pd.DataFrame(prediction_records)
    prediction_columns = [*POINT_KEYS, "prediction", "forecast_month", "prediction_date"]
    if predictions.empty:
        predictions = pd.DataFrame(columns=prediction_columns)
    _required_columns(predictions, prediction_columns, "Candidate predictions")
    predictions = _canonical_keys(predictions[prediction_columns])
    predictions["month_index"] = pd.to_numeric(predictions["month_index"], errors="coerce")
    predictions = predictions.loc[predictions["month_index"].notna() & predictions["month_index"].mod(1).eq(0)].copy()
    predictions["month_index"] = predictions["month_index"].astype("int64")
    if predictions.duplicated(POINT_KEYS).any():
        raise ValueError("Candidate has duplicate canonical game/task/month predictions.")
    predictions["prediction"] = pd.to_numeric(predictions["prediction"], errors="coerce")
    predictions["forecast_month"] = _dates(predictions["forecast_month"], month=True)
    predictions["prediction_date"] = _dates(predictions["prediction_date"])
    predictions = predictions.rename(columns={
        "prediction": "_raw_prediction", "forecast_month": "_candidate_forecast_month",
        "prediction_date": "_candidate_prediction_date",
    })
    predictions["_candidate_row_present"] = True
    points = points.merge(predictions, on=POINT_KEYS, how="left", validate="one_to_one")
    calendar_match = (
        (points["forecast_month"] == points["_candidate_forecast_month"])
        & (points["prediction_date"] == points["_candidate_prediction_date"])
    )
    numeric_prediction = pd.to_numeric(points["_raw_prediction"], errors="coerce")
    points["prediction"] = numeric_prediction.where(calendar_match & np.isfinite(numeric_prediction))
    present = points["_candidate_row_present"].eq(True)
    points["prediction_missing_reason"] = np.where(
        ~present, "missing_point", np.where(~calendar_match, "calendar_or_prediction_date_mismatch",
                                            np.where(~np.isfinite(numeric_prediction), "nonfinite_prediction", ""))
    )
    points["error"] = points["prediction"] - points["actual"]
    points["absolute_error"] = points["error"].abs()
    points["squared_error"] = points["error"] ** 2
    points["direction_correct"] = np.nan
    scored = points.loc[points["prediction"].notna()].sort_values([*SERIES_KEYS, "month_index"])
    for _, group in scored.groupby(SERIES_KEYS, sort=False, dropna=False):
        consecutive = group["month_index"].diff().eq(1)
        direction = np.sign(group["prediction"].diff()).eq(np.sign(group["actual"].diff())).astype(float)
        points.loc[group.index, "direction_correct"] = direction.where(consecutive)
    for key, value in metadata.items():
        points[key] = value

    def summarize(group, result_level, bracket):
        paired = group.loc[group["prediction"].notna()]
        expected = len(group)
        paired_count = len(paired)
        expected_games = group["target_game_key"].nunique()
        game_counts = paired.groupby("target_game_key")["month_index"].count()
        games_meeting_minimum = int((game_counts >= min_months).sum())
        row = {key: group.iloc[0][key] for key in [*TASK_KEYS, "evaluation_split"]}
        per_game = result_level == "per_game"
        row.update({
            **metadata, "result_level": result_level, "bracket": bracket,
            "_target_id": int(group.iloc[0]["_target_id"]) if per_game else None,
            "target_game_key": str(group.iloc[0]["target_game_key"]) if per_game else None,
            "target_game": str(group.iloc[0]["target_game"]) if per_game else None,
            "expected_games": int(expected_games), "scored_games": int(len(game_counts)),
            "games_meeting_min_months": games_meeting_minimum,
            "expected_months": expected, "valid_months": paired_count,
            "missing_forecast_months": expected - paired_count,
            "forecast_coverage": paired_count / expected if expected else 0.0,
            "directional_comparisons": int(paired["direction_correct"].count()),
            "mae": float(paired["absolute_error"].mean()) if paired_count else None,
            "rmse": float(np.sqrt(paired["squared_error"].mean())) if paired_count else None,
            "bias": float(paired["error"].mean()) if paired_count else None,
            "directional_accuracy": float(paired["direction_correct"].mean()) if paired["direction_correct"].notna().any() else None,
            "aggregation": "pooled_observed_months",
        })
        row["eligible_for_selection"] = bool(
            result_level == "overall" and row["evaluation_split"] == "validation"
            and expected > 0 and paired_count == expected
            and games_meeting_minimum == expected_games
            and row["mae"] is not None and math.isfinite(row["mae"])
            and row["rmse"] is not None and math.isfinite(row["rmse"])
        )
        return row

    metrics = []
    for _, group in points.groupby([*TASK_KEYS, "evaluation_split"], sort=True, dropna=False):
        metrics.append(summarize(group, "overall", "all_games"))
        for bracket, subset in group.groupby("actual_bracket", sort=True, dropna=False):
            metrics.append(summarize(subset, "bracket", str(bracket)))
        for _, subset in group.groupby("target_game_key", sort=True, dropna=False):
            metrics.append(summarize(subset, "per_game", str(subset.iloc[0]["actual_bracket"])))
    metrics_df = pd.DataFrame(metrics)
    # Pandas would otherwise coerce nullable aggregate IDs to float and lose
    # precision for Spark xxhash64 IDs. Preserve Python integers explicitly.
    if not metrics_df.empty:
        metrics_df["_target_id"] = pd.Series([row["_target_id"] for row in metrics], dtype="object")
    return metrics_df, points


def select_validation_winners(metrics_records):
    """Rank only full-coverage validation-overall MAE, then RMSE and run ID."""
    frame = metrics_records.copy() if isinstance(metrics_records, pd.DataFrame) else pd.DataFrame(metrics_records)
    required = [*TASK_KEYS, "forecast_run_id", "result_level", "evaluation_split",
                "eligible_for_selection", "forecast_coverage", "expected_months", "valid_months", "mae", "rmse"]
    _required_columns(frame, required, "Candidate metrics")
    board = frame.loc[(frame["result_level"] == "overall") & (frame["evaluation_split"] == "validation")].copy()
    if board.duplicated([*TASK_KEYS, "forecast_run_id"]).any():
        raise ValueError("Each candidate needs exactly one validation-overall row per task.")
    board["validation_rank"] = pd.Series(pd.NA, index=board.index, dtype="Int64")
    board["is_recommended"] = False
    board["selection_reason"] = "incomplete_validation_coverage"
    board["selection_status"] = "not_selected"
    numeric_mae = pd.to_numeric(board["mae"], errors="coerce")
    numeric_rmse = pd.to_numeric(board["rmse"], errors="coerce")
    eligible = board["eligible_for_selection"].map(_true)
    eligible &= board["forecast_coverage"].eq(1.0) & board["valid_months"].eq(board["expected_months"])
    eligible &= board["expected_months"].gt(0) & np.isfinite(numeric_mae) & np.isfinite(numeric_rmse)
    finite_metrics = np.isfinite(numeric_mae) & np.isfinite(numeric_rmse)
    board.loc[~finite_metrics, "selection_reason"] = "no_finite_validation_metrics"
    board.loc[eligible, "selection_reason"] = "ranked_by_validation_mae"
    for _, group in board.loc[eligible].groupby(TASK_KEYS, sort=True, dropna=False):
        ordered = group.sort_values(["mae", "rmse", "forecast_run_id"], kind="stable")
        board.loc[ordered.index, "validation_rank"] = list(range(1, len(ordered) + 1))
        board.loc[ordered.index[0], "is_recommended"] = True
        board.loc[ordered.index[0], "selection_status"] = "recommended_validation"
    board = board.sort_values([*TASK_KEYS, "validation_rank", "forecast_run_id"], na_position="last").reset_index(drop=True)
    winners = board.loc[board["is_recommended"]].copy().reset_index(drop=True)
    return board, winners


In [0]:
# Build or reuse immutable labels, then evaluate candidates one at a time.
if public_evaluation_settings() != LOADED_EVALUATION_SETTINGS:
    raise ValueError("Settings changed after loading. Rerun the loading cell first.")

COHORT_POINT_COLUMNS = [
    *POINT_KEYS, "_target_id", "target_game", "game_release_date", "prediction_date", "forecast_month",
    "actual", "reference_observed_months", "split", "evaluation_split", "actual_bracket",
    "bracket_actual", "bracket_percent_rank", "bracket_month",
]


def json_scalar(value):
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (date, datetime, pd.Timestamp)):
        return value.isoformat()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def normalized_records(records):
    return [{key: json_scalar(value) for key, value in row.items()} for row in records]


def records_signature(records):
    encoded = sorted(hashlib.sha256(canonical_json(row).encode()).hexdigest() for row in normalized_records(records))
    return {"row_count": len(encoded), "sha256": hashlib.sha256("\n".join(encoded).encode()).hexdigest()}


if SAVED_COHORT_RECORDS is None:
    reference_columns = [*POINT_KEYS, "_target_id", "target_game", "game_release_date", "prediction_date", "forecast_month",
                         "actual", "actual_is_observed", "actual_contains_imputed_history", "target_curve_starts_before_prediction"]
    reference_records = [row.asDict() for row in long_all_df.where(
        F.col("forecast_run_id") == REFERENCE_RUN_ID
    ).select(*reference_columns).toLocalIterator()]
    cohort_points_pdf, game_cohort_pdf = build_reference_cohort(
        reference_records, evaluation_months=EVALUATION_MONTHS, min_months=MIN_MONTHS,
        bracket_month=BRACKET_MONTH, val_fraction=VALIDATION_FRACTION, test_fraction=TEST_FRACTION,
        exclude_pre_release=EXCLUDE_PRE_RELEASE_CURVES,
        exclude_cumulative_imputed_history=EXCLUDE_CUMULATIVE_IMPUTED_HISTORY,
    )
    COHORT_CONFIG = {"policy": cohort_policy(), "forecast_compatibility": REFERENCE_COMPATIBILITY,
                     "cohort_implementation_version": "fixed_actual_cohort_v2"}
else:
    # Strip persistence metadata: the engine must receive canonical labels only.
    cohort_points_pdf = pd.DataFrame(SAVED_COHORT_RECORDS)[COHORT_POINT_COLUMNS].copy()
    for name in ["game_release_date", "prediction_date", "forecast_month"]:
        cohort_points_pdf[name] = pd.to_datetime(cohort_points_pdf[name], errors="coerce").dt.date
    games = cohort_points_pdf[["target_game_key", "_target_id", "target_game", "game_release_date"]].drop_duplicates()
    if games["target_game_key"].duplicated().any():
        raise ValueError("Saved cohort contains conflicting canonical-game identities.")
    game_cohort_pdf = pd.DataFrame(assign_temporal_splits(games.to_dict("records"), VALIDATION_FRACTION, TEST_FRACTION))
    expected_splits = game_cohort_pdf.set_index("target_game_key")["split"]
    if not cohort_points_pdf["split"].eq(cohort_points_pdf["target_game_key"].map(expected_splits)).all():
        raise ValueError("Saved cohort split membership does not match its immutable policy.")
    COHORT_CONFIG = SAVED_COHORT_CONFIG

cohort_points_pdf = cohort_points_pdf[COHORT_POINT_COLUMNS].copy()
if cohort_points_pdf.duplicated(POINT_KEYS).any():
    raise ValueError("Frozen cohort has duplicate expected points.")
COHORT_CONFIG_JSON = canonical_json(COHORT_CONFIG)
COHORT_CONFIG_HASH = hashlib.sha256(COHORT_CONFIG_JSON.encode()).hexdigest()
COHORT_CONTENT_HASH = records_signature(cohort_points_pdf.to_dict("records"))["sha256"]
RESOLVED_COHORT_ID = "cohort_" + hashlib.sha256((COHORT_CONFIG_JSON + COHORT_CONTENT_HASH).encode()).hexdigest()[:24]
if SAVED_COHORT_RECORDS is not None:
    stored_hashes = {row["cohort_content_hash"] for row in SAVED_COHORT_RECORDS}
    if stored_hashes != {COHORT_CONTENT_HASH} or RESOLVED_COHORT_ID != COHORT_ID.strip():
        raise ValueError("Saved cohort content differs from its recorded hash or ID.")

EVALUATION_ID = "eval_" + uuid.uuid4().hex
EVALUATION_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
EVALUATION_CONFIG = {
    **copy.deepcopy(LOADED_EVALUATION_SETTINGS),
    "selected_forecast_run_ids": list(SELECTED_RUN_IDS), "reference_forecast_run_id": REFERENCE_RUN_ID,
    "cohort_id": RESOLVED_COHORT_ID, "cohort_config_hash": COHORT_CONFIG_HASH,
    "cohort_content_hash": COHORT_CONTENT_HASH,
    "input_table_versions": copy.deepcopy(INPUT_TABLE_VERSIONS),
    "forecast_config_hashes": {run_id: RUN_METADATA[run_id]["forecast_config_hash"] for run_id in SELECTED_RUN_IDS},
}
EVALUATION_CONFIG_JSON = canonical_json(EVALUATION_CONFIG)
EVALUATION_CONFIG_HASH = hashlib.sha256(EVALUATION_CONFIG_JSON.encode()).hexdigest()
EVALUATION_PARENT_MLFLOW_RUN_ID = None
EVALUATION_CHILD_RUN_IDS = {}
EVALUATION_SAVE_IDENTITY = None
EVALUATED_RUN_CONFIGS = copy.deepcopy({run_id: RUN_CONFIGS[run_id] for run_id in SELECTED_RUN_IDS})


def assert_evaluation_settings():
    if public_evaluation_settings() != LOADED_EVALUATION_SETTINGS or canonical_json(EVALUATION_CONFIG) != EVALUATION_CONFIG_JSON:
        raise ValueError("Evaluation settings changed. Rerun loading/evaluation for a new evaluation ID.")


metric_frames = []
ALIGNMENT_DIAGNOSTICS = {}
for run_id in SELECTED_RUN_IDS:
    prediction_columns = [*POINT_KEYS, "prediction", "forecast_month", "prediction_date"]
    predictions = [row.asDict() for row in long_all_df.where(
        F.col("forecast_run_id") == run_id
    ).select(*prediction_columns).toLocalIterator()]
    frame, scored_points = evaluate_forecast_run(predictions, cohort_points_pdf, MIN_MONTHS, RUN_METADATA[run_id])
    metric_frames.append(frame)
    ALIGNMENT_DIAGNOSTICS[run_id] = {
        str(reason): int(count) for reason, count in scored_points["prediction_missing_reason"].value_counts().items()
        if reason
    }
forecast_error_metrics_pdf = pd.concat(metric_frames, ignore_index=True)
forecast_error_metrics_pdf["evaluation_id"] = EVALUATION_ID
forecast_error_metrics_pdf["evaluation_config_hash"] = EVALUATION_CONFIG_HASH
forecast_error_metrics_pdf["cohort_id"] = RESOLVED_COHORT_ID
forecast_error_metrics_pdf["split_version"] = RESOLVED_COHORT_ID
forecast_error_metrics_pdf["game_id"] = forecast_error_metrics_pdf["_target_id"].astype("object")
forecast_error_metrics_pdf["created_at_utc"] = EVALUATION_STARTED_AT_UTC
leaderboard_pdf, recommendations_pdf = select_validation_winners(forecast_error_metrics_pdf)

selected_records = []
for row in recommendations_pdf.to_dict("records"):
    run_id = row["forecast_run_id"]
    selected_records.append({
        "evaluation_id": EVALUATION_ID, "evaluation_config_hash": EVALUATION_CONFIG_HASH,
        "cohort_id": RESOLVED_COHORT_ID, "split_version": RESOLVED_COHORT_ID,
        **{key: row[key] for key in TASK_KEYS},
        "selected_forecast_run_id": run_id, "selected_experiment_id": row["experiment_id"],
        "selected_forecast_mlflow_run_id": row["forecast_mlflow_run_id"],
        "selected_forecast_config_hash": row["forecast_config_hash"],
        "selected_forecast_config_json": canonical_json(RUN_CONFIGS[run_id]),
        "validation_mae": row["mae"], "validation_rmse": row["rmse"], "validation_bias": row["bias"],
        "validation_forecast_coverage": row["forecast_coverage"],
        "expected_validation_months": row["expected_months"], "expected_validation_games": row["expected_games"],
        "selection_status": "recommended_validation", "selection_policy": "minimum_validation_mae_full_coverage",
        "created_at_utc": EVALUATION_STARTED_AT_UTC,
    })

# Explicit schemas preserve null aggregates and full 64-bit game IDs.
POINT_SPEC = [
    ("cohort_id", "str"), ("cohort_config_hash", "str"), ("cohort_config_json", "str"), ("cohort_content_hash", "str"),
    *[(name, "str") for name in [*TASK_KEYS, "target_game_key"]], ("month_index", "long"),
    ("_target_id", "long"), ("target_game", "str"), ("game_release_date", "date"),
    ("prediction_date", "date"), ("forecast_month", "date"), ("actual", "double"),
    ("reference_observed_months", "long"), ("split", "str"), ("evaluation_split", "str"),
    ("actual_bracket", "str"), ("bracket_actual", "double"), ("bracket_percent_rank", "double"), ("bracket_month", "long"),
]
GAME_SPEC = [
    ("cohort_id", "str"), ("cohort_config_hash", "str"), ("target_game_key", "str"),
    ("_target_id", "long"), ("target_game", "str"), ("game_release_date", "date"), ("split", "str"),
    ("test_cutoff_date", "date"), ("validation_cutoff_date", "date"), ("eligible_games_in_cohort", "long"),
]
METRIC_SPEC = [
    *[(name, "str") for name in ["evaluation_id", "evaluation_config_hash", "cohort_id", "split_version",
        "forecast_run_id", "forecast_mlflow_run_id", "forecast_config_hash", "experiment_id", "experiment_name",
        "similarity_mlflow_run_id", "similarity_config_hash", *TASK_KEYS, "evaluation_split", "result_level", "bracket"]],
    ("_target_id", "long"), ("game_id", "long"), ("target_game_key", "str"), ("target_game", "str"),
    *[(name, "long") for name in ["expected_games", "scored_games", "games_meeting_min_months", "expected_months",
                                   "valid_months", "missing_forecast_months", "directional_comparisons"]],
    *[(name, "double") for name in ["forecast_coverage", "mae", "rmse", "bias", "directional_accuracy"]],
    ("eligible_for_selection", "bool"), ("aggregation", "str"), ("created_at_utc", "str"),
]
SELECTED_SPEC = [
    *[(name, "str") for name in ["evaluation_id", "evaluation_config_hash", "cohort_id", "split_version", *TASK_KEYS,
        "selected_forecast_run_id", "selected_experiment_id", "selected_forecast_mlflow_run_id",
        "selected_forecast_config_hash", "selected_forecast_config_json"]],
    *[(name, "double") for name in ["validation_mae", "validation_rmse", "validation_bias", "validation_forecast_coverage"]],
    ("expected_validation_months", "long"), ("expected_validation_games", "long"),
    ("selection_status", "str"), ("selection_policy", "str"), ("created_at_utc", "str"),
]


def typed_records(records, specification):
    result = []
    for record in records:
        row = {}
        for name, kind in specification:
            value = json_scalar(record.get(name))
            if value is None:
                row[name] = None
            elif kind == "long":
                row[name] = int(value)
            elif kind == "double":
                row[name] = float(value)
            elif kind == "bool":
                row[name] = _true(value)
            elif kind == "date":
                row[name] = pd.Timestamp(value).date()
            else:
                row[name] = str(value)
        result.append(row)
    return result


def spark_records(records, specification):
    kinds = {"str": StringType(), "long": LongType(), "double": DoubleType(), "bool": BooleanType(), "date": DateType()}
    schema = StructType([StructField(name, kinds[kind], True) for name, kind in specification])
    return spark.createDataFrame(typed_records(records, specification), schema=schema)


cohort_records = cohort_points_pdf.to_dict("records")
for row in cohort_records:
    row.update(cohort_id=RESOLVED_COHORT_ID, cohort_config_hash=COHORT_CONFIG_HASH,
               cohort_config_json=COHORT_CONFIG_JSON, cohort_content_hash=COHORT_CONTENT_HASH)
game_records = game_cohort_pdf.to_dict("records")
for row in game_records:
    row.update(cohort_id=RESOLVED_COHORT_ID, cohort_config_hash=COHORT_CONFIG_HASH)
OUTPUT_SPECS = {"cohort": POINT_SPEC, "splits": GAME_SPEC, "metrics": METRIC_SPEC, "selected": SELECTED_SPEC}
OUTPUT_RECORDS = {
    "cohort": typed_records(cohort_records, POINT_SPEC),
    "splits": typed_records(game_records, GAME_SPEC),
    "metrics": typed_records(forecast_error_metrics_pdf.to_dict("records"), METRIC_SPEC),
    "selected": typed_records(selected_records, SELECTED_SPEC),
}
OUTPUT_TABLES = {"cohort": COHORTS_TABLE, "splits": SPLITS_TABLE, "metrics": METRICS_TABLE, "selected": SELECTED_TABLE}
OUTPUT_SIGNATURES = {label: records_signature(records) for label, records in OUTPUT_RECORDS.items()}
PREPARED_EVALUATION_ID = EVALUATION_ID

print(f"Evaluation ID: {EVALUATION_ID}")
print(f"Frozen cohort ID: {RESOLVED_COHORT_ID}")
print("Reuse this COHORT_ID to evaluate later configurations against the same labels and splits.")
display(spark_records(game_records, GAME_SPEC).groupBy("split").count())
if leaderboard_pdf.empty:
    print("There are no validation games under this split; no configuration is recommended.")
else:
    display(spark_records(leaderboard_pdf.to_dict("records"), METRIC_SPEC + [
        ("validation_rank", "long"), ("is_recommended", "bool"), ("selection_reason", "str"), ("selection_status", "str"),
    ]))
    if not selected_records:
        print("No candidate covers every expected validation point. Inspect forecast_coverage; no winner was forced.")
print("Evaluation is prepared. Run the next cell to preserve results and log MLflow.")


In [0]:
# Preserve fixed cohorts and evaluation results; retry safely after partial saves.
import platform
import time
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import Param, Metric

assert_evaluation_settings()
if PREPARED_EVALUATION_ID != EVALUATION_ID or list(SELECTED_RUN_IDS) != EVALUATION_CONFIG["selected_forecast_run_ids"]:
    raise ValueError("Evaluation outputs are stale. Rerun evaluation before saving.")
current_signatures = {label: records_signature(records) for label, records in OUTPUT_RECORDS.items()}
if current_signatures != OUTPUT_SIGNATURES:
    raise ValueError("Prepared evaluation output changed. Rerun evaluation for a new ID.")
save_identity = (EVALUATION_ID, EVALUATION_CONFIG_HASH, canonical_json(OUTPUT_SIGNATURES))
if EVALUATION_SAVE_IDENTITY is not None and EVALUATION_SAVE_IDENTITY != save_identity:
    raise ValueError("This evaluation ID is already associated with different output content.")


def stored_evaluation_rows(table, key_name, key_value):
    if not spark.catalog.tableExists(table):
        return None
    frame = spark.table(table)
    if key_name not in frame.columns:
        return None
    return frame.where(F.col(key_name) == key_value)


def collect_records(frame, columns):
    if frame is None:
        return []
    return [row.asDict() for row in frame.select(*columns).toLocalIterator()]


# Recover tracking identities from a previous successful metrics write.
previous_metrics = stored_evaluation_rows(METRICS_TABLE, "evaluation_id", EVALUATION_ID)
if previous_metrics is not None:
    identities = previous_metrics.select(
        "forecast_run_id", "evaluation_mlflow_run_id", "evaluation_parent_mlflow_run_id"
    ).distinct().collect()
    parents = {row["evaluation_parent_mlflow_run_id"] for row in identities}
    if None in parents or len(parents) > 1:
        raise RuntimeError("Conflicting evaluation-parent identities in saved metrics.")
    if parents:
        parent_id = next(iter(parents))
        if EVALUATION_PARENT_MLFLOW_RUN_ID is not None and EVALUATION_PARENT_MLFLOW_RUN_ID != parent_id:
            raise RuntimeError("A different tracking parent already owns this evaluation ID.")
        EVALUATION_PARENT_MLFLOW_RUN_ID = parent_id
    for row in identities:
        forecast_id, child_id = row["forecast_run_id"], row["evaluation_mlflow_run_id"]
        if not child_id or (forecast_id in EVALUATION_CHILD_RUN_IDS and EVALUATION_CHILD_RUN_IDS[forecast_id] != child_id):
            raise RuntimeError("Conflicting evaluation-child identities in saved metrics.")
        EVALUATION_CHILD_RUN_IDS[forecast_id] = child_id

client = MlflowClient()
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_PATH)
if experiment is None:
    try:
        experiment_id = client.create_experiment(MLFLOW_EXPERIMENT_PATH)
    except Exception:
        experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_PATH)
        if experiment is None:
            raise
        experiment_id = experiment.experiment_id
else:
    experiment_id = experiment.experiment_id

if EVALUATION_PARENT_MLFLOW_RUN_ID is None:
    EVALUATION_PARENT_MLFLOW_RUN_ID = client.create_run(experiment_id, tags={
        "mlflow.runName": f"evaluation_{EVALUATION_ID[-8:]}", "pipeline_stage": "evaluation_batch",
        "evaluation_id": EVALUATION_ID, "cohort_id": RESOLVED_COHORT_ID,
    }).info.run_id
parent_id = EVALUATION_PARENT_MLFLOW_RUN_ID
EVALUATION_SAVE_IDENTITY = save_identity


def append_evaluation_output(label):
    shared_cohort = label in {"cohort", "splits"}
    key_name, key_value = ("cohort_id", RESOLVED_COHORT_ID) if shared_cohort else ("evaluation_id", EVALUATION_ID)
    table, specification = OUTPUT_TABLES[label], OUTPUT_SPECS[label]
    columns = [name for name, _ in specification]
    expected = OUTPUT_SIGNATURES[label]
    existing = stored_evaluation_rows(table, key_name, key_value)
    old_records = collect_records(existing, columns) if existing is not None else []
    if old_records:
        if records_signature(old_records) != expected:
            raise RuntimeError(f"Immutable {label} content already differs for {key_value}.")
    else:
        records = copy.deepcopy(OUTPUT_RECORDS[label])
        write_spec = list(specification)
        if not shared_cohort:
            forecast_field = "forecast_run_id" if label == "metrics" else "selected_forecast_run_id"
            for row in records:
                row["evaluation_mlflow_run_id"] = EVALUATION_CHILD_RUN_IDS[row[forecast_field]]
                row["evaluation_parent_mlflow_run_id"] = parent_id
            write_spec += [("evaluation_mlflow_run_id", "str"), ("evaluation_parent_mlflow_run_id", "str")]
        frame = spark_records(records, write_spec)
        (frame.write.format("delta").mode("append").option("mergeSchema", "true")
         .option("txnAppId", f"evaluation:{key_value}:{label}").option("txnVersion", 0)
         .option("userMetadata", json.dumps({key_name: key_value, "output": label}))
         .saveAsTable(table))
    persisted = stored_evaluation_rows(table, key_name, key_value)
    if persisted is None or records_signature(collect_records(persisted, columns)) != expected:
        raise RuntimeError(f"{label} content differs after save.")
    if not shared_cohort and expected["row_count"]:
        forecast_field = "forecast_run_id" if label == "metrics" else "selected_forecast_run_id"
        saved_ids = persisted.select(forecast_field, "evaluation_mlflow_run_id", "evaluation_parent_mlflow_run_id").distinct().collect()
        if any(row["evaluation_parent_mlflow_run_id"] != parent_id
               or row["evaluation_mlflow_run_id"] != EVALUATION_CHILD_RUN_IDS[row[forecast_field]] for row in saved_ids):
            raise RuntimeError(f"{label} belongs to different evaluation tracking runs.")
    commit = (
        spark.sql(f"DESCRIBE HISTORY {quoted_table_name(table)}")
        .where(F.get_json_object("userMetadata", f"$.{key_name}") == key_value)
        .agg(F.min("version").alias("v")).first()["v"]
    )
    return {"table": table, "delta_commit_version": None if commit is None else int(commit), **expected}


try:
    client.log_dict(parent_id, EVALUATION_CONFIG, "evaluation_configuration.json")
    client.log_dict(parent_id, COHORT_CONFIG, "cohort_policy.json")
    client.log_dict(parent_id, ALIGNMENT_DIAGNOSTICS, "alignment_diagnostics.json")
    client.log_dict(parent_id, {"python": platform.python_version(), "pandas": pd.__version__,
        "numpy": np.__version__, "spark": spark.version, "mlflow": mlflow.__version__,
        "implementation_version": IMPLEMENTATION_VERSION}, "environment.json")
    for forecast_id in SELECTED_RUN_IDS:
        if forecast_id not in EVALUATION_CHILD_RUN_IDS:
            EVALUATION_CHILD_RUN_IDS[forecast_id] = client.create_run(experiment_id, tags={
                "mlflow.parentRunId": parent_id,
                "mlflow.runName": f"evaluate_{forecast_id[-12:]}", "pipeline_stage": "forecast_evaluation",
                "evaluation_id": EVALUATION_ID, "cohort_id": RESOLVED_COHORT_ID,
                "forecast_run_id": forecast_id,
                "upstream_forecast_mlflow_run_id": RUN_METADATA[forecast_id]["forecast_mlflow_run_id"],
            }).info.run_id
        child_id = EVALUATION_CHILD_RUN_IDS[forecast_id]
        config = EVALUATED_RUN_CONFIGS[forecast_id]
        params = {
            "forecast_run_id": forecast_id, "similarity_experiment_id": config["similarity_experiment_id"],
            "cohort_id": RESOLVED_COHORT_ID, "evaluation_months": EVALUATION_MONTHS,
            "minimum_observed_months": MIN_MONTHS, "bracket_month": BRACKET_MONTH,
            "distance_method": config["distance_method"], "distance_cutoff": config["distance_cutoff"],
            "minimum_neighbors": config["minimum_neighbors"], "parent_game_process": config["parent_game_process"],
        }
        params.update({f"weight_{group}": value for group, value in config["feature_group_weights"].items()})
        params.update({f"source_version_{label}": value for label, value in config["source_versions_resolved"].items()})
        metric_entities = []
        for row in OUTPUT_RECORDS["metrics"]:
            if row["forecast_run_id"] != forecast_id or row["result_level"] == "per_game":
                continue
            level = "overall" if row["result_level"] == "overall" else "bracket_" + row["bracket"]
            prefix = "/".join([row["evaluation_split"], row["forecast_type"], row["revenue_type"], row["own_status"], row["metric_name"], level])
            for name in ["mae", "rmse", "bias", "directional_accuracy", "forecast_coverage", "valid_months",
                         "expected_months", "expected_games", "scored_games", "directional_comparisons"]:
                value = row[name]
                if value is not None and math.isfinite(float(value)):
                    metric_entities.append(Metric(f"{prefix}/{name}", float(value), int(time.time() * 1000), 0))
        client.log_batch(child_id, params=[Param(key, str(value)) for key, value in params.items()], synchronous=True)
        for offset in range(0, len(metric_entities), 500):
            client.log_batch(child_id, metrics=metric_entities[offset:offset + 500], synchronous=True)
        client.log_dict(child_id, config, "forecast_configuration.json")
        client.log_dict(child_id, ALIGNMENT_DIAGNOSTICS[forecast_id], "alignment_diagnostics.json")
        client.set_tag(child_id, "validation_recommended", str(any(
            row["selected_forecast_run_id"] == forecast_id for row in OUTPUT_RECORDS["selected"])).lower())

    locations = {label: append_evaluation_output(label) for label in ["cohort", "splits", "metrics", "selected"]}
    client.log_dict(parent_id, locations, "results/delta_outputs.json")
    completion = {
        "evaluation_id": EVALUATION_ID, "evaluation_config_hash": EVALUATION_CONFIG_HASH,
        "evaluation_parent_mlflow_run_id": parent_id,
        "evaluation_child_run_ids_json": canonical_json(EVALUATION_CHILD_RUN_IDS),
        "cohort_id": RESOLVED_COHORT_ID, "configuration_json": EVALUATION_CONFIG_JSON,
        "output_signatures_json": canonical_json(OUTPUT_SIGNATURES),
        "output_locations_json": canonical_json(locations), "status": "complete",
        "started_at_utc": EVALUATION_STARTED_AT_UTC,
    }
    previous = stored_evaluation_rows(EVALUATIONS_TABLE, "evaluation_id", EVALUATION_ID)
    if previous is None or not previous.limit(1).count():
        (spark.createDataFrame([completion]).write.format("delta").mode("append").option("mergeSchema", "true")
         .option("txnAppId", f"evaluation:{EVALUATION_ID}:complete").option("txnVersion", 0)
         .option("userMetadata", json.dumps({"evaluation_id": EVALUATION_ID, "output": "complete"}))
         .saveAsTable(EVALUATIONS_TABLE))
    saved = stored_evaluation_rows(EVALUATIONS_TABLE, "evaluation_id", EVALUATION_ID).limit(2).collect()
    if len(saved) != 1 or any(saved[0][name] != completion[name] for name in [
        "evaluation_config_hash", "evaluation_parent_mlflow_run_id", "evaluation_child_run_ids_json", "cohort_id",
        "configuration_json", "output_signatures_json", "status",
    ]):
        raise RuntimeError("The saved evaluation completion record differs from this evaluation.")
    for child_id in EVALUATION_CHILD_RUN_IDS.values():
        client.set_tag(child_id, "evaluation_save_status", "complete")
        client.set_terminated(child_id, status="FINISHED")
    client.set_tag(parent_id, "evaluation_save_status", "complete")
    client.set_terminated(parent_id, status="FINISHED")
except BaseException as error:
    for run_id in [*EVALUATION_CHILD_RUN_IDS.values(), parent_id]:
        try:
            client.set_tag(run_id, "evaluation_save_status", "failed")
            client.set_tag(run_id, "failure_type", type(error).__name__)
            client.set_terminated(run_id, status="KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
        except Exception:
            pass
    raise

print(f"Saved evaluation: {EVALUATION_ID}")
print(f"Cohort to reuse: {RESOLVED_COHORT_ID}")
print(f"MLflow parent run: {parent_id}")
print(f"Validation recommendations: {len(OUTPUT_RECORDS['selected'])}")
print("The final chart cell is optional; metrics and recommendations are already saved.")


In [0]:
# Complete metrics table: filter evaluation_split/result_level in the table UI.
# This displays the current evaluation; it does not recompute or save anything.
forecast_error_metrics_df = spark_records(OUTPUT_RECORDS["metrics"], METRIC_SPEC)
if forecast_error_metrics_df.head(1): display(
    forecast_error_metrics_df.select(
        "evaluation_split", "experiment_name", "forecast_type", "metric_name",
        "revenue_type", "own_status", "result_level", "bracket", "target_game",
        "mae", "rmse", "bias", "directional_accuracy", "forecast_coverage",
        "valid_months", "expected_months", "scored_games", "expected_games",
        "directional_comparisons", "forecast_run_id", "evaluation_id", "_target_id",
    ).orderBy(
        "evaluation_split", "forecast_type", "revenue_type", "metric_name",
        "forecast_run_id", "result_level", "bracket", "target_game",
    )
)


In [0]:
# Optional review: one test game, one Sale/Rent channel, one forecast run.
# The exact retained lists come from 06; no distance cutoff is recalculated here.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def build_review_options(game_records, available_records, run_metadata, channel_records):
    """Test-only choices; unavailable predictions remain reviewable."""
    games = {str(row["target_game_key"]): dict(row) for row in game_records
             if row.get("split") == "test"}
    channels = {(str(row["target_game_key"]), row["revenue_type"])
                for row in channel_records if row.get("split") == "test"}
    options = {}
    labels = {"sales": "Sale", "lease": "Rent"}
    for item in available_records:
        row = dict(item)
        key, run_id, channel = str(row["target_game_key"]), row["forecast_run_id"], row["revenue_type"]
        if key not in games or (key, channel) not in channels or channel not in labels or run_id not in run_metadata:
            continue
        game, metadata = games[key], run_metadata[run_id]
        label = f"{game['target_game']} | {labels[channel]} (test) | {run_id}"
        options[label] = {
            "forecast_run_id": run_id, "target_game_key": key,
            "_target_id": int(game["_target_id"]), "target_game": str(game["target_game"]),
            "revenue_type": channel, "own_status": "owned" if channel == "sales" else "leased",
            "experiment_id": metadata.get("experiment_id"), "split": "test",
        }
    return dict(sorted(options.items()))


def reference_curve(points_pdf, context, forecast_type, metric_name):
    if points_pdf.empty:
        return points_pdf.copy()
    frame = points_pdf.loc[
        points_pdf["target_game_key"].eq(context["target_game_key"])
        & points_pdf["revenue_type"].eq(context["revenue_type"])
        & points_pdf["own_status"].eq(context["own_status"])
        & points_pdf["forecast_type"].eq(forecast_type)
        & points_pdf["metric_name"].eq(metric_name)
    ]
    return frame.sort_values("month_index").copy()


def numeric_curve(frame, column):
    if frame.empty or column not in frame:
        return np.full(len(frame), np.nan, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype=float)


def review_forecast_timing(frame, frozen_actuals, value_columns):
    """Mask calendar/origin mismatches at months with frozen evaluation labels."""
    if frame.empty or frozen_actuals.empty or "forecast_month" not in frozen_actuals:
        return frame, 0
    prepared = frame.copy()
    if "forecast_month" in prepared:
        forecast_dates = pd.to_datetime(prepared["forecast_month"], errors="coerce")
    elif "forecast_yearmonth" in prepared:
        forecast_dates = pd.to_datetime(prepared["forecast_yearmonth"].astype("string") + "01",
                                        format="%Y%m%d", errors="coerce")
    elif "curve_start_month" in prepared:
        starts = pd.to_datetime(prepared["curve_start_month"], errors="coerce")
        forecast_dates = pd.Series([date + pd.DateOffset(months=int(month) - 1) if pd.notna(date) else pd.NaT
                                   for date, month in zip(starts, prepared["month_index"])], index=prepared.index)
    else:
        forecast_dates = pd.Series(pd.NaT, index=prepared.index, dtype="datetime64[ns]")
    references = frozen_actuals.set_index("month_index")
    reference_dates = pd.to_datetime(prepared["month_index"].map(references["forecast_month"]), errors="coerce")
    has_reference = prepared["month_index"].isin(references.index)
    mismatch = has_reference & (forecast_dates.isna() | reference_dates.isna() | forecast_dates.ne(reference_dates))
    if "prediction_date" in references:
        reference_origins = pd.to_datetime(prepared["month_index"].map(references["prediction_date"]), errors="coerce")
        origins = (pd.to_datetime(prepared["prediction_date"], errors="coerce") if "prediction_date" in prepared
                   else pd.Series(pd.NaT, index=prepared.index, dtype="datetime64[ns]"))
        mismatch |= has_reference & (origins.isna() | reference_origins.isna() | origins.ne(reference_origins))
    for column in value_columns:
        if column in prepared:
            prepared.loc[mismatch, column] = np.nan
    return prepared, int(mismatch.sum())


def plot_review_panel(ax, frame, frozen_actuals, prediction_column, lower_column,
                      upper_column, title, ylabel, color, horizon):
    ax.set_title(title, loc="left")
    ax.set_xlabel("Month since first source observation")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(1, horizon + 1))
    ax.grid(alpha=0.2)
    frame, mismatches = review_forecast_timing(frame, frozen_actuals, [prediction_column, lower_column, upper_column])
    if mismatches:
        ax.text(0.02, 0.75, f"{mismatches} calendar/origin mismatch(es) excluded", transform=ax.transAxes, va="top", fontsize=9)
    if not frame.empty:
        x = numeric_curve(frame, "month_index")
        predicted = numeric_curve(frame, prediction_column)
        if np.isfinite(predicted).any():
            ax.plot(x, predicted, color=color, marker="o", label="Saved forecast")
            ax.fill_between(x, numeric_curve(frame, lower_column), numeric_curve(frame, upper_column),
                            color=color, alpha=0.14, label="Neighbor variation (±2 SD)")
        else:
            ax.text(0.02, 0.95, "No usable neighbor forecast", transform=ax.transAxes, va="top")
    else:
        ax.text(0.02, 0.95, "No saved series for this run", transform=ax.transAxes, va="top")
    if not frozen_actuals.empty:
        ax.plot(numeric_curve(frozen_actuals, "month_index"), numeric_curve(frozen_actuals, "actual"),
                color="#222222", marker="D", linestyle="--", label="Frozen observed actual")
    else:
        ax.text(0.02, 0.85, "No observed labels in this evaluation cohort", transform=ax.transAxes, va="top")
    handles, _ = ax.get_legend_handles_labels()
    if handles:
        ax.legend(fontsize=9)


def plot_channel_review(demand_pdf, performance_pdf, reference_points, context, metrics, horizon):
    """Saved predictions with the same frozen labels used by the metrics."""
    channel = "Sale" if context["revenue_type"] == "sales" else "Rent"
    color = "#1768AC" if context["revenue_type"] == "sales" else "#B45F06"
    fig, axes = plt.subplots(1 + len(metrics), 1, figsize=(11, 4.0 * (1 + len(metrics))), squeeze=False)
    axes = axes[:, 0]
    demand_metric = None
    if not demand_pdf.empty:
        demand_metric = str(demand_pdf["metric_name"].iloc[0])
    else:
        matching = reference_points.loc[
            reference_points["target_game_key"].eq(context["target_game_key"])
            & reference_points["revenue_type"].eq(context["revenue_type"])
            & reference_points["forecast_type"].eq("demand")
        ]
        if not matching.empty:
            demand_metric = str(matching["metric_name"].iloc[0])
    demand_title = "Cumulative sales" if context["revenue_type"] == "sales" else "Monthly rental records"
    plot_review_panel(
        axes[0], demand_pdf, reference_curve(reference_points, context, "demand", demand_metric),
        "forecast_value", "low_band_2std", "high_band_2std", demand_title, demand_title, color, horizon,
    )
    titles = {"avg_coin_in_index_vs_house": "Coin-in index vs. house",
              "avg_theo_net_win_index_vs_house": "Theo net win index vs. house"}
    for ax, metric in zip(axes[1:], metrics):
        plot_review_panel(
            ax, performance_pdf, reference_curve(reference_points, context, "performance", metric),
            f"{metric}_mean", f"{metric}_lower_2std", f"{metric}_higher_2std",
            f"{titles.get(metric, metric)} — {context['own_status']}", "Index", color, horizon,
        )
    fig.suptitle(f"{context['target_game']} | {channel} | TEST\nForecast run: {context['forecast_run_id']}", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    return fig


def reconstruct_neighbor_performance(source_pdf, metrics, horizon):
    """Selected as-of rows only; reproduce 06's slot-bucket weighting and fills."""
    columns = ["neighbor_game", "_neighbor_key", "month_index", *[f"{metric}_mean" for metric in metrics]]
    if source_pdf.empty:
        return pd.DataFrame(columns=columns)
    source = source_pdf.copy()
    source["_perf_month"] = pd.to_datetime(source["_perf_month"], errors="coerce")
    source["no_of_slots"] = pd.to_numeric(source["no_of_slots"], errors="coerce")
    source = source.loc[source["_perf_month"].notna() & source["no_of_slots"].gt(0)].copy()
    for metric in metrics:
        source[metric] = pd.to_numeric(source[metric], errors="coerce").replace([np.inf, -np.inf], np.nan)
    output = []
    for neighbor_key, neighbor in source.groupby("_neighbor_key", sort=True):
        neighbor = neighbor.copy()
        start = neighbor["_perf_month"].min()
        neighbor["month_index"] = ((neighbor["_perf_month"].dt.year - start.year) * 12
                                   + neighbor["_perf_month"].dt.month - start.month + 1)
        last = min(int(neighbor["month_index"].max()), int(horizon))
        observed = neighbor.loc[neighbor["month_index"].between(1, horizon)]
        by_slots = observed.groupby(["month_index", "no_of_slots"], as_index=False)[metrics].mean()
        curve = pd.DataFrame({"month_index": range(1, last + 1)})
        curve["neighbor_game"] = str(neighbor["neighbor_game"].iloc[0])
        curve["_neighbor_key"] = neighbor_key
        for metric in metrics:
            valid = by_slots.loc[by_slots[metric].notna()].copy()
            valid["_weighted"] = valid[metric] * valid["no_of_slots"]
            stats = valid.groupby("month_index")[["no_of_slots", "_weighted"]].sum()
            stats = stats.reindex(curve["month_index"]).where(lambda frame: frame["no_of_slots"].gt(0)).ffill()
            curve[f"{metric}_mean"] = (stats["_weighted"] / stats["no_of_slots"]).to_numpy(dtype=float)
        output.append(curve)
    return pd.concat(output, ignore_index=True)[columns] if output else pd.DataFrame(columns=columns)


def plot_eilers_neighbor_review(context, retained_neighbors_df, target_performance_pdf, reference_points):
    config = RUN_CONFIGS[context["forecast_run_id"]]
    neighbors = (
        retained_neighbors_df.where(F.col("neighbor_type") == "eilers")
        .select(normalized_name(F.col("matched_source_name")).alias("_neighbor_key"), "neighbor_game")
        .distinct()
    )
    if not neighbors.limit(1).count():
        print("This run has no retained Eilers neighbors for the selected channel.")
        return
    if target_performance_pdf.empty or "prediction_date" not in target_performance_pdf:
        print("No prediction date is available for the optional neighbor chart.")
        return
    dates = pd.to_datetime(target_performance_pdf["prediction_date"], errors="coerce").dropna()
    if dates.empty:
        print("Unknown prediction date: this series has no historical neighbor inputs.")
        return
    prediction_date = dates.iloc[0].date()
    source = (
        spark.read.option("versionAsOf", config["source_versions_resolved"]["performance"])
        .table(config["source_paths"]["performance"])
        .withColumn("_neighbor_key", normalized_name(F.col("game_name")))
        .withColumn("_status", normalized_name(F.col("own_status")))
        .withColumn("_perf_month", F.trunc(F.to_date("yearmonth", "yyyy-MM-dd"), "month"))
        .withColumn("no_of_slots", finite_double(F.col("no_of_slots")))
        .where((F.col("_status") == context["own_status"]) & (F.col("no_of_slots") > 0)
               & (F.col("_perf_month") < F.trunc(F.lit(prediction_date), "month")))
        .join(neighbors, "_neighbor_key", "inner")
    )
    if config["slot_counts"]:
        source = source.where(F.col("no_of_slots").isin(config["slot_counts"]))
    source = source.select("neighbor_game", "_neighbor_key", "_perf_month", "no_of_slots", *[
        finite_double(F.col(metric)).alias(metric) for metric in performance
    ])
    # Only the selected game's retained sources have reached the driver.
    monthly = reconstruct_neighbor_performance(source.toPandas(), performance, EVALUATION_MONTHS)
    if monthly.empty:
        print("The retained Eilers neighbors have no observations available at prediction time.")
        return
    fig, axes = plt.subplots(len(performance), 1, figsize=(11, 4.0 * len(performance)), squeeze=False)
    for ax, metric in zip(axes[:, 0], performance):
        actuals = reference_curve(reference_points, context, "performance", metric)
        for name, frame in monthly.groupby("neighbor_game", sort=True):
            ax.plot(frame["month_index"], frame[f"{metric}_mean"], alpha=0.65, label=name)
        if not target_performance_pdf.empty:
            target, _ = review_forecast_timing(target_performance_pdf, actuals, [f"{metric}_mean"])
            ax.plot(numeric_curve(target, "month_index"),
                    numeric_curve(target, f"{metric}_mean"), color="black", linestyle="--",
                    linewidth=2, label="Saved target forecast")
        if not actuals.empty:
            ax.plot(numeric_curve(actuals, "month_index"), numeric_curve(actuals, "actual"),
                    color="#B42525", marker="D", linestyle=":", label="Frozen observed actual")
        ax.set_title(f"{metric} — {context['own_status']}", loc="left")
        ax.set_xlabel("Month since each game's first Eilers observation")
        ax.set_xticks(range(1, EVALUATION_MONTHS + 1))
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)
    fig.suptitle(f"{context['target_game']} | Retained historical Eilers neighbors\n{context['forecast_run_id']}", fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


# Available rows, including null forecasts, determine which saved runs can be reviewed.
available_review_records = [row.asDict() for row in (
    demand_all_df.select("forecast_run_id", "target_game_key", "revenue_type")
    .unionByName(performance_all_df.select("forecast_run_id", "target_game_key", "revenue_type"))
    .distinct().collect()
)]
game_option_lookup = build_review_options(
    game_cohort_pdf.to_dict("records"), available_review_records, RUN_METADATA,
    cohort_points_pdf[["target_game_key", "revenue_type", "split"]].drop_duplicates().to_dict("records"),
)
# Cascading selectors update immediately without rerunning evaluation or saving.
import ipywidgets as widgets
from IPython.display import display, clear_output

# Dispose of callbacks and controls from a previous execution of this cell.
for old_widget in globals().get("review_widgets", []):
    old_widget.close()


def build_review_index(contexts, run_metadata):
    """Experiment -> unique test game/channel -> saved forecast runs."""
    index = {}
    for context in contexts:
        run_id = context["forecast_run_id"]
        metadata = run_metadata[run_id]
        experiment_id = str(metadata.get("experiment_id") or "Unknown experiment")
        experiment_name = str(metadata.get("experiment_name") or experiment_id)
        experiment = index.setdefault(experiment_id, {"name": experiment_name, "games": {}})
        game_key = (context["target_game_key"], context["revenue_type"])
        experiment["games"].setdefault(game_key, {})[run_id] = context
    return index


review_index = build_review_index(game_option_lookup.values(), RUN_METADATA)
experiment_selector = widgets.Dropdown(description="Experiment:", layout=widgets.Layout(width="95%"),
                                       style={"description_width": "100px"})
game_selector = widgets.Dropdown(description="Test game:", layout=widgets.Layout(width="95%"),
                                 style={"description_width": "100px"})
run_selector = widgets.Dropdown(description="Forecast run:", layout=widgets.Layout(width="95%"),
                                style={"description_width": "100px"})
review_output = widgets.Output()
review_controls = widgets.VBox([experiment_selector, game_selector, run_selector, review_output])
review_widgets = [experiment_selector, game_selector, run_selector, review_output, review_controls]
_review_updating = False
selected_context = None


def selected_game_context():
    experiment = review_index.get(experiment_selector.value, {})
    return experiment.get("games", {}).get(game_selector.value, {}).get(run_selector.value)


def render_selected_review():
    global selected_context, selected_demand_pdf, selected_performance_pdf
    selected_context = selected_game_context()
    with review_output:
        clear_output(wait=True)
        try:
            _plot_selected_review(selected_context)
        except Exception as error:
            print(f"Game review unavailable: {type(error).__name__}: {error}")


def update_review_runs():
    experiment = review_index.get(experiment_selector.value, {})
    runs = experiment.get("games", {}).get(game_selector.value, {})
    previous = run_selector.value
    run_selector.options = [(run_id, run_id) for run_id in sorted(runs)]
    if previous in runs:
        run_selector.value = previous
    run_selector.disabled = len(runs) <= 1
    run_selector.layout.display = "" if len(runs) > 1 else "none"


def on_review_experiment(change):
    global _review_updating
    if _review_updating:
        return
    _review_updating = True
    try:
        experiment = review_index.get(experiment_selector.value, {})
        choices = []
        for game_key, runs in experiment.get("games", {}).items():
            context = next(iter(runs.values()))
            channel = "Sale" if context["revenue_type"] == "sales" else "Rent"
            label = f"{context['target_game']} | {channel} (test) | ID {context['_target_id']}"
            choices.append((label, game_key))
        previous = game_selector.value
        game_selector.options = sorted(choices, key=lambda item: item[0].casefold())
        if previous in experiment.get("games", {}):
            game_selector.value = previous
        game_selector.disabled = not choices
        update_review_runs()
    finally:
        _review_updating = False
    render_selected_review()


def on_review_game(change):
    global _review_updating
    if _review_updating:
        return
    _review_updating = True
    try:
        update_review_runs()
    finally:
        _review_updating = False
    render_selected_review()


def on_review_run(change):
    if not _review_updating:
        render_selected_review()


def _plot_selected_review(selected_context):
    global selected_demand_pdf, selected_performance_pdf
    selected_demand_pdf = pd.DataFrame()
    selected_performance_pdf = pd.DataFrame()
    if selected_context is not None:
        condition = (
            (F.col("forecast_run_id") == selected_context["forecast_run_id"])
            & (F.col("target_game_key") == selected_context["target_game_key"])
            & (F.col("revenue_type") == selected_context["revenue_type"])
            & (F.col("own_status") == selected_context["own_status"])
            & F.col("month_index").between(1, EVALUATION_MONTHS)
        )
        selected_demand_pdf = demand_all_df.where(condition).orderBy("month_index").toPandas()
        selected_performance_pdf = performance_all_df.where(condition).orderBy("month_index").toPandas()
        plot_channel_review(selected_demand_pdf, selected_performance_pdf, cohort_points_pdf,
                            selected_context, performance, EVALUATION_MONTHS)
        plt.show()
        selected_neighbors_df = forecast_neighbors_df.where(
            (F.col("forecast_run_id") == selected_context["forecast_run_id"])
            & (F.col("target_game_key") == selected_context["target_game_key"])
            & (F.col("revenue_type") == selected_context["revenue_type"])
            & (F.col("own_status") == selected_context["own_status"])
        )
        print("Exact retained neighbor lists from this forecast run (including unavailable-as-of sources):")
        display(selected_neighbors_df.orderBy("neighbor_type", "neighbor_rank"))
        if SHOW_NEIGHBOR_CHARTS:
            try:
                plot_eilers_neighbor_review(selected_context, selected_neighbors_df,
                                            selected_performance_pdf, cohort_points_pdf)
            except Exception as error:
                print(f"Optional neighbor chart unavailable: {type(error).__name__}: {error}")
        else:
            print("Set SHOW_NEIGHBOR_CHARTS=True to plot the retained historical Eilers curves.")

    if selected_context is None:
        print("No saved test series belongs to the selected experiment and fixed evaluation cohort.")


# Remove the legacy combined dropdown so it no longer clutters the toolbar.
try:
    dbutils.widgets.remove("test_game")
except Exception:
    pass

name_counts = {}
for experiment in review_index.values():
    name_counts[experiment["name"]] = name_counts.get(experiment["name"], 0) + 1
experiment_choices = [
    (experiment["name"] if name_counts[experiment["name"]] == 1
     else f"{experiment['name']} | {experiment_id}", experiment_id)
    for experiment_id, experiment in review_index.items()
]
experiment_selector.options = sorted(experiment_choices, key=lambda item: item[0].casefold())
experiment_selector.disabled = not experiment_choices
experiment_selector.observe(on_review_experiment, names="value")
game_selector.observe(on_review_game, names="value")
run_selector.observe(on_review_run, names="value")
display(review_controls)
on_review_experiment(None)
